Task 1: File ingestion and regular expressions
- The first function in this task is used to identify the encoding of the file that is read. Since these files don't have any spescial character (for example æ, ø or å), the filter won't necessarily find the "True" encoding. When using the generate_data_for_d2 code the files created with the Latin-1 encoding gets labled as UTF-8, further strenghtening the idea that identifing the encoding will be difficult. The function identifies the encoding using try/except. if decoding the file using UTF-8 returns a Unicode-error, we know the encoding is Latin-1. This code works fine as long as the only encodings are UTF-8 or Latin-1, but if there was another encoding it would get labled as Latin-1 and not it's actual type. Since we know that there are only two encodings in this file-system, it will work, but it is something to keep in mind if we were to use it for another purpose.

- The second function uses the data from the tuple made by the previous function and sorts the different elements into a dictionary for each line. These dictinaries are then added to a list. These regular expressions are quite long and spessific. They work for this code, but they require that we know what each line looks like. If a line were to start with the [timestamp], the filtering would fail. This is somthing to keep in mind if we were to work further on the code. 

In [423]:
from pathlib import Path
import re


def file_reader(file_path: Path) -> tuple:
    """The function reads checks the encoding of the file.
    It then reads the file and puts the file data and encoding into a tuple."""
    if not Path(file_path).exists():
        raise Exception(f"File Does not exist {file_path}")
    raw = Path(file_path).read_bytes()
    try:
        raw.decode("utf-8")
        enc = "utf-8"
    except UnicodeDecodeError:
        enc = "iso-8859-1"
    with open(file_path, mode="r", encoding=enc) as f:
        data = f.read()
    return (data, enc)


def log_parser(data: list) -> list[dict]:
    """The function removes the noise lines.
    Then identifies the different the different elements of interest using regex.
    Each line is then added to its on dictionary, and collected in one list."""
    valid_lines = []
    lines = data.split("\n")
    # remove noise lines
    for line in lines:

        if re.match(r"\[\d+\-\d+\-\d+ \d+\:\d+\:\d+\]", line) is not None:
            timestamp = re.search(r"\d+\-\d+\-\d+\s\d+\:\d+\:\d+", line).group(0)
            register_id = re.search(r"\w+\_\d+", line).group(0)
            item = re.search(r"(item=)(\w+)", line).group(2)
            price = re.search(r"\d+\.\d+", line).group(0)
            currency = re.search(r"(\s)(\w{3})(\s)", line).group(2)
            status = re.search(r"(\()(\w+)(\))", line).group(2)

            times = {"timestamp": timestamp,
                     "register_id": register_id,
                     "item": item,
                     "price": float(price),
                     "currency": currency,
                     "status": status
                     }
            valid_lines.append(times)

    return valid_lines


Task 2: Recursive utility functions
- The first function traverses the filesystem by calling itself. When the function reaches a file of interrest. These files are then added to a list. To do this we used the iterdir()-function which iterates over all the files in a directory. If the element is a directory, the function calls itself to walk further into the file-system. When it reaches a filetype it's searching for it appends the path to a list.

- The second function goes through the json-file(s) in the file-system. It checks if the part of the json it is "pointing" at has a key called "registers". If so it retrives the id-value. If it isn't "pointing" at doesnt match what we want, it calls itself to move the "pointer". 

- The third function calls the second function and stores the output i a set. A set could be a more usefull container than a list in this case since all the elements are unique, and we are just interrested in the unique registers, and not multiples of the same register.


In [424]:
from pathlib import Path
import json

def traverse_file_system(current_path: Path, path_list: list, file_suffix=".log") -> list[Path]:
    """Traverses the file system and appends the files of a certain type into a list"""
    p = Path(current_path)
    for dir in p.iterdir():
        if dir.is_dir():
            traverse_file_system(dir, path_list, file_suffix)
        else:
            if dir.suffix == file_suffix:
                path_list.append(dir)
    return path_list


def walk_active_registers(node: json) -> dict:
    """Walks through the register ids in the file system"""
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "registers" and isinstance(value, list):
                for i, reg in enumerate(value):
                    if reg["active"]:
                        yield (reg["id"])
            else:
                yield from walk_active_registers(value)
    elif isinstance(node, list):
        for i, value in enumerate(node):
            yield from walk_active_registers(value)


def find_active_registers(node) -> set[str]:
    """Calls the previous function at puts the register id-values into a set"""
    return set(walk_active_registers(node))


Task 3: Functions for cleaning and analysing data using NumPy 
- The first function goes through every dictionary i the list containing the elements retrived from a file. It then uses multiple functions from the NumPy to calculate the SD-value, mean and median. The valid price-values are added to a list. We then use a for-loop to go through every element in the parameter-list and replace the outlier-values with the median.

- The second function calculates the refund ratio. 10**-6 is added to avoid zero-divition, this could also be done by using an if/else statment where the equation is only used if the sales_count isn't zero. For the purpose of this program this change makes little difference.

In [425]:
import numpy as np


def remove_outliers(price_lines: list[dict]) -> np.ndarray:
    """The funtion removes the price outliers and replaces them with the median value.
    The prices are then added to a list."""
    item_prices = []
    for i in price_lines:
        item_price = float(i["price"])
        item_prices.append(item_price)
    item_prices_array = np.array(item_prices)

    sd = np.std(item_prices_array)
    mean_price = np.mean(item_prices_array)
    prices_no_outliers = []
    valid = [i for i in item_prices if np.abs(i - mean_price) <= 3 * sd]
    median_price = np.median(valid)

    for p in item_prices:
        if np.abs(p - mean_price) > 3 * sd:
            prices_no_outliers.append(median_price)
        else:
            prices_no_outliers.append(p)
    final_prices = np.array(prices_no_outliers)

    return final_prices


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates the refund ratio."""
    refund_ratio = refund_count / (sales_count + 10**-6)
    return refund_ratio


Task 4: Data analysis using Pandas
- The first function concerts the ".log"-file to a dataframe with the elements of interest. The time is set as the index so that we can sort the dataframe by it (ascending order). To be able to do this we converted the timestamp-value to datetime. If we wanted to save some space, we could retrive the values in the dictionary instead of asigning the to variables first. 

- The second function uses groupby() to split, apply and combine data. It creates a table with a human-friendly presentation of the data.

- The third function creates a spreadsheet-style pivot table, whith the items as the rows and the transaction status as the columns. 

- The fourth function creates a table of the refund ratios per item using the compute_refund_ratio and put the results into a dictionary.

In [426]:
import pandas as pd


def log_to_dataframe(log: list[dict]) -> pd.DataFrame:
    """The function converts a list of dictionaries to a DataFrame with the wanted elements"""
    transaction_time = [pd.to_datetime(p["timestamp"]) for p in log]
    transaction_price = [float(p["price"]) for p in log]
    transaction_item = [p["item"] for p in log]
    transaction_status = [p["status"] for p in log]
    transaction_currency = [p["currency"] for p in log]

    transaction_df = pd.DataFrame({"time": transaction_time,
                  "item": transaction_item,
                  "price": transaction_price,
                  "currency": transaction_currency,
                  "status": transaction_status})
    transaction_df = transaction_df.set_index("time")
    transaction_df = transaction_df.sort_values("time")

    return transaction_df


def sum_total_per_item_per_hour(dataframe: pd.DataFrame) -> pd.DataFrame:
    """The function groups the data by the 'item'-key.
    It the calculates the sum for each item per hour"""
    grouped_items = dataframe.groupby(["item", pd.Grouper(freq="1h")])[["price"]].sum()
    return grouped_items


def pivot_by_transaction_status(dataframe: pd.DataFrame) -> pd.DataFrame:
    """The function pivots a dataframe and creates a table with the statuses as columns and items as rows."""
    pivot_table = dataframe.pivot_table(index="item", columns="status", values="price", aggfunc="sum")
    return pivot_table


def refund_ratio_per_item(dataframe: pd.DataFrame) -> list[dict]:
    """The funtion pivots the dataframe and adds the values of interrest to a NumPy array.
    This array is used to calculate the refund ratio (also bu using the compute_refund_ration-function).
    These values are returned in a list of dictionaries."""
    counts = dataframe.pivot_table(index="item", columns="status", values="price", aggfunc="count", fill_value=0)
    success_counts = counts["SUCCESS"].to_numpy()
    refund_counts = counts["REFUND"].to_numpy()
    refund_ratio = compute_refund_ratio(success_counts, refund_counts)

    refund_ratio_list = []
    for i, item in enumerate(counts.index):
        refund_ratio_list.append({"item": item,
                                  "refund_ratio": refund_ratio[i]})
    return refund_ratio_list


Task 5: Create full workflow
- The first thing that was done was to retrive the information we wanted using the functions we created in task 1 - 4.

- The written functions are combined to write a set of files. To do this we used first used st_options to make sure the print-statements printed the whole tables to the files.

    - To make the first and second csv file we used pandas' to_csv method.

    - To make the JSON file we used pandas' to_json method.
    
    - To create the last csv file we used the open() function since this data was not in a usable format for the to_csv method.

In [427]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

all_json_files = traverse_file_system("Data", [], ".json")

all_active_registers = []
for json_file in all_json_files:
    json_config = json.loads(Path(json_file).read_text(encoding="utf-8"))
    active_registers = find_active_registers(json_config)
    all_active_registers.extend(active_registers)

all_log_files = traverse_file_system("Data", [], ".log")

all_sales_information = []
for log_file in all_log_files:
    values = file_reader(log_file)
    sales_information = log_parser(values[0])
    all_sales_information.extend(sales_information)

cleaned_prices = remove_outliers(all_sales_information)
for record, price in zip(all_sales_information, cleaned_prices):
    record["price"] = price

dataframe = log_to_dataframe(all_sales_information)
dataframe.loc[dataframe["currency"] == "NOK", "price"] *= 10
dataframe["price"] = dataframe["price"].round(2)

dataframe.to_csv("cleaned_transactions.csv", mode="w")

resampled = sum_total_per_item_per_hour(dataframe)
resampled["price"].unstack("item").to_json("hourly_revenue.json", date_format="iso", indent=4, mode="w")

pivoted_data = pivot_by_transaction_status(dataframe)
pivoted_data.to_csv("status_pivot_table.csv", mode="w")

refund_ratio_data = refund_ratio_per_item(dataframe)
with open("item_refund_ratios.csv", mode="w+") as f:
    print("item, refund_ratio", file=f)
    for e in refund_ratio_data:
        print(f"{e['item']}, {e['refund_ratio']:.2f}", file=f)
